![Kenya Food Price Early Warning System](images/banner.png)

# Kenya Food Price Early Warning System


Forecasting staple food prices across Kenyan markets to give farmers, traders, and food security actors the early signal they currently lack.

## 1. Business Understanding

## 1.1 Background

Food prices in Kenya are highly seasonal and can change sharply across markets. Staples such as maize and beans are affected by harvest cycles, rainfall, supply conditions, and limited access to timely market information.

For farmers, this creates a difficult decision: sell early and risk missing a better price, or wait and risk a price drop. Traders and institutions face a similar problem when deciding when to buy, store, release, or distribute food.

The problem is not a lack of historical data. Kenya has publicly available market price data, but this information is mainly used to understand what has already happened.

This project aims to turn that historical information, together with weather data, into a forward-looking system that shows **where prices are likely to move and when unusual price changes may be starting.**

## 1.2 Problem Statement

Farmers, traders, and food security institutions often make decisions using current or historical prices rather than reliable forecasts of what may happen next.

This can lead to poor selling and buying decisions, increased inventory risk, and delayed responses to food price shocks. Institutions may only act once a shortage or price increase is already visible.

The core problem is therefore not the absence of data. It is the lack of a system that combines historical prices and external factors such as weather to produce **market-specific forecasts and early warnings.**

## 1.3 Business Objectives

The project aims to:

1. **Provide forward-looking price visibility**  
   Forecast commodity prices 2–3 months ahead for individual markets.

2. **Detect emerging price shocks**  
   Identify when actual prices begin to move significantly away from expected prices.

3. **Improve access to market intelligence**  
   Present forecasts and trends through a simple interactive dashboard that non-technical users can understand.

4. **Support institutional decisions**  
   Provide quantitative signals that can support decisions around food reserves, subsidies, procurement, and humanitarian response.

5. **Build a reproducible system**  
   Create an automated pipeline from data collection and processing to forecasting and dashboard deployment.

### 1.4 Stakeholder Analysis

The system is designed for people who make decisions around food prices, from farmers and traders to government and humanitarian organizations.

**Smallholder Farmers and Cooperatives**
Need to decide when to sell, how much to sell, and whether holding stock could lead to a better return. Price forecasts can give them greater visibility into upcoming market conditions.

**Traders and Market Intermediaries**
Need to decide when and where to buy, store, and sell commodities. Forecasts can help them manage inventory and reduce the risk of buying at a peak or holding through a price decline.

**County Agricultural Offices and NDMA**
Need early signals of localized food price stress. Market-level forecasts and anomaly alerts can help them identify emerging risks before they become wider food security problems.

**NGOs and Humanitarian Organizations**
Need to plan procurement and cash-based interventions efficiently. Earlier visibility into price movements can help them act before rising prices reduce the purchasing power of their interventions.

**National Cereals and Produce Board**
Needs to make better decisions around strategic reserves, procurement, and price stabilization. Forecasts can provide an additional signal when deciding when to buy or release stock.

**Urban Consumers and Low-Income Households**
Are highly sensitive to changes in staple food prices. Earlier information about expected price movements can help households and organizations supporting them prepare for potential increases.

**Food Processors and Millers**
Need predictable input costs to manage production and pricing. Forward-looking commodity prices can support better procurement and planning.

Across these groups, the common need is **timely, market-specific information about where food prices are heading**. The forecasting and anomaly detection components are designed to provide that signal.


## 1.5 Business Success Criteria

The project will be considered successful if:

- Forecasts perform better than a simple baseline such as the last observed price or seasonal average.
- The anomaly detection system identifies meaningful price shocks without producing excessive false alarms.
- A non-technical user can select a market and commodity and quickly understand the expected price, forecast range, and alert status.
- The entire data pipeline is reproducible and can be refreshed without significant manual work.
- The system uses reliable and accessible public data sources so it can be extended beyond the capstone.

## 1.6 Data Mining Goals

The data science work will focus on five main tasks:

1. Build a **Prophet forecasting model** as the baseline.
2. Build an **LSTM model** for comparison.
3. Combine WFP Kenya food price data with NASA POWER weather data using market location and date.
4. Build a **residual-based anomaly detection system** to identify unusual price movements.
5. Compare the models using **MAE and MAPE** across commodities and markets.

The final forecasts and alerts will be presented through an interactive **Streamlit dashboard**.

## 1.7 Data Mining Success Criteria

The technical implementation will be considered successful if:

- The forecasting models outperform the chosen naive baseline across most market-commodity combinations.
- Forecast accuracy is evaluated using MAE and MAPE.
- The anomaly detector identifies historical price shocks while limiting false alerts.
- The price and weather datasets can be joined with minimal data loss.
- The pipeline can run end-to-end with minimal manual intervention.
- The deployed dashboard works reliably for the selected markets and commodities.

## 1.8 Hypotheses Guiding the Analysis

The analysis will test six hypotheses:

- **H1:** Maize prices follow a seasonal pattern around harvest periods.
- **H2:** Rainfall has a measurable lagged relationship with future commodity prices.
- **H3:** Price patterns differ significantly between markets.
- **H4:** Maize and bean prices show a relationship because they are commonly used as staple food substitutes.
- **H5:** Price volatility increases during drought periods.
- **H6:** Changes in wholesale prices are reflected in retail prices within one to two weeks.

These hypotheses will guide the exploratory analysis, feature engineering, modelling, and evaluation throughout the project.

## 2. Data Understanding

This section is highly  focused on understanding the data before cleaning, joining, and modelling.

The project uses two main data sources: the **WFP Kenya Food Prices dataset**, sourced through HDX, and daily weather data from the **NASA POWER API**.

The goal is to understand what each dataset contains, assess its quality, and look for early patterns that can help test the hypotheses from the Business Understanding phase.


### 2.1 Data Collection

The project uses two publicly accessible data sources.

**WFP Kenya Food Prices (HDX)**
Provides historical commodity prices by market, commodity, and date. This is the primary source for the target variable used in forecasting.

**NASA POWER API**
Provides daily weather observations, including rainfall and temperature. These variables are used as external features to test whether weather conditions can improve price forecasts.

The extraction date and source versions are recorded to ensure that the analysis can be reproduced using the same data snapshot.


In [1]:
# core libraries and notebook display settings
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests # used for routing and fetching our data and later in in demo(front end)
from datetime import datetime



In [2]:
# set display settings for the notebook

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 150)
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)

In [3]:
# record the date at which the last data was pulled.

extraction_date = datetime.now().strftime("%Y-%m-%d")

print(f"Data extraction date recorded: {extraction_date}")


Data extraction date recorded: 2026-09-01


In [4]:
# Load the Kenya food prices dataset. Add support for Kaggle and local loading seamlessly.

DATA_URL = (
    "https://data.humdata.org/dataset/"
    "e0d3fba6-f9a2-45d7-b949-140c455197ff/"
    "resource/517ee1bf-2437-4f8c-aa1b-cb9925b9d437/"
    "download/wfp_food_prices_ken.csv"
)

# define a filename
FILENAME = "wfp_food_prices_ken.csv"


def load_food_prices():
    """
    Load the Kenya food prices dataset.

    The function automatically checks for a Kaggle copy,
    a previously saved local copy, or the original source URL.

    It returns:
        pandas.DataFrame
            Raw Kenya food prices dataset.
    """

    # check if the notebook is running on Kaggle
    kaggle_root = "/kaggle/input"

    if os.path.exists(kaggle_root):

        # search for the dataset inside the Kaggle input directory
        for root, _, files in os.walk(kaggle_root):

            if FILENAME in files:

                kaggle_path = os.path.join(root, FILENAME)

                try:
                    prices_raw = pd.read_csv(kaggle_path)

                    print(f"Loaded dataset from Kaggle: {kaggle_path}")

                    return prices_raw

                except Exception as error:
                    print(f"Kaggle file could not be read: {error}")

    # check if a previously downloaded local copy exists
    if os.path.exists(FILENAME):

        try:
            prices_raw = pd.read_csv(FILENAME)

            print(f"Loaded local dataset: {FILENAME}")

            return prices_raw

        except Exception as error:
            print(f"Local file could not be read: {error}")

    # try downloading from the source URL
    print("Dataset not found locally. Trying the source URL...")

    try:
        response = requests.get(
            DATA_URL,
            timeout=30
        )

        response.raise_for_status()

        # save the downloaded file locally for future use
        with open(FILENAME, "wb") as file:
            file.write(response.content)

        prices_raw = pd.read_csv(FILENAME)

        print(f"Dataset downloaded and saved as '{FILENAME}'")

        return prices_raw

    except Exception as error:
        print(f"Download failed: {error}")

    # final fallback to the local file
    if os.path.exists(FILENAME):

        print("Using the previously saved local dataset.")

        return pd.read_csv(FILENAME)

    raise FileNotFoundError(
        "Could not load the Kenya food prices dataset. "
        "Check your internet connection or provide a local copy."
    )


In [5]:
# Load the dataset
prices_raw = load_food_prices()

# Output the first 5 rows
prices_raw.head()

Loaded local dataset: wfp_food_prices_ken.csv


,date,admin1,admin2,market,market_id,latitude,longitude,category,commodity,commodity_id,unit,priceflag,pricetype,currency,price,usdprice
0,2006-01-15,Coast,Mombasa,Mombasa,191,-4.05,39.67,cereals and tubers,Maize (white),67,90 KG,actual,Wholesale,KES,1480.00,20.58
1,2006-01-15,Coast,Mombasa,Mombasa,191,-4.05,39.67,pulses and nuts,Beans,50,KG,actual,Wholesale,KES,33.63,0.47
2,2006-01-15,Coast,Mombasa,Mombasa,191,-4.05,39.67,pulses and nuts,Beans (dry),262,90 KG,actual,Wholesale,KES,3246.00,45.15
3,2006-01-15,Eastern,Kitui,Kitui,187,-1.37,38.02,cereals and tubers,Maize (white),67,KG,actual,Retail,KES,17.00,0.24
4,2006-01-15,Eastern,Kitui,Kitui,187,-1.37,38.02,cereals and tubers,Potatoes (Irish),148,50 KG,actual,Wholesale,KES,1249.99,17.39


In [6]:
# drop the units/description row if present, then fix dtypes 

def clean_price_data(prices_raw):
    """Clean and convert data types in the raw price dataset."""

    # Remove the units/description row if present
    if prices_raw.iloc[0].astype(str).str.startswith("#").any():

        prices = prices_raw.iloc[1:].reset_index(drop=True)

    else:

        prices = prices_raw.copy()

    # Convert date and price columns to the correct data types
    prices["date"] = pd.to_datetime(
        prices["date"],
        errors="coerce"
    )

    prices["price"] = pd.to_numeric(
        prices["price"],
        errors="coerce"
    )

    # Convert USD price if the column exists
    if "usdprice" in prices.columns:

        prices["usdprice"] = pd.to_numeric(
            prices["usdprice"],
            errors="coerce"
        )

    return prices


In [7]:
prices = clean_price_data(prices_raw)

In [8]:
print(f"Rows: {prices.shape[0]}, Columns: {prices.shape[1]}")
print(f"Date range: {prices['date'].min()} to {prices['date'].max()}")

Rows: 28062, Columns: 16
Date range: 2006-01-15 00:00:00 to 2026-08-15 00:00:00


In [9]:
# Get a sample of daily weather data from the NASA POWER API.
def get_weather_sample(
    latitude,
    longitude,
    start="20240101",
    end="20240131"
):
    """

    It returns:
        dict
            Weather parameters returned by the API.
    """

    power_url = "https://power.larc.nasa.gov/api/temporal/daily/point"

    params = {
        "parameters": "T2M,T2M_MAX,T2M_MIN,PRECTOTCORR,RH2M",
        "community": "ag",
        "longitude": longitude,
        "latitude": latitude,
        "start": start,
        "end": end,
        "format": "JSON",
    }

    # ping the url and pass a timeout to avoid out of control requests
    response = requests.get(
        power_url,
        params=params,
        timeout=30
    )

    # raise an error if the API request failed
    response.raise_for_status()
    print(f" status code {response.status_code}")


    power_sample = response.json()

    return power_sample["properties"]["parameter"]




In [10]:
# sample call for Nairobi to confirm the API and response structure
nairobi_lat, nairobi_lon = -1.2864, 36.8172

sample_params = get_weather_sample(
    latitude=nairobi_lat,
    longitude=nairobi_lon
)


 status code 200


In [11]:

print("Parameters returned:", list(sample_params.keys()))
print("Sample PRECTOTCORR values:", list(sample_params["PRECTOTCORR"].items())[:5])

Parameters returned: ['T2M', 'T2M_MAX', 'T2M_MIN', 'PRECTOTCORR', 'RH2M']
Sample PRECTOTCORR values: [('20240101', 0.31), ('20240102', 1.38), ('20240103', 0.28), ('20240104', 1.51), ('20240105', 1.56)]


### NASA POWER Weather Variables

The NASA POWER API provides several weather variables that can be used to understand conditions that may influence commodity prices.

| Parameter | Description | Unit |
|---|---|---|
| `T2M` | Average air temperature measured 2 meters above the surface | °C |
| `T2M_MAX` | Maximum air temperature measured at 2 meters | °C |
| `T2M_MIN` | Minimum air temperature measured at 2 meters | °C |
| `T2M_RANGE` | Difference between the daily maximum and minimum temperature | °C |
| `RH2M` | Relative humidity measured at 2 meters | % |
| `WS2M` | Average wind speed measured at 2 meters | m/s |
| `PS` | Atmospheric pressure at the surface | kPa |
| `PRECTOTCORR` | Bias-corrected total precipitation | mm/day |
| `ALLSKY_SFC_SW_DWN` | Total solar radiation reaching the surface under all sky conditions | kW-hr/m²/day |

For this project, **rainfall and temperature** are the primary weather variables of interest because they have the strongest potential relationship with agricultural production and future commodity prices.

### 2.2 Data Description

Before combining the datasets, we first need to understand how each one is structured.

The **price dataset** contains one observation for a specific commodity, market, and date.

The **weather dataset** contains daily weather observations for a specific geographic location.

This structure allows the two datasets to be joined using **market location and date**.


In [12]:
# column level summary: dtype, uniqueness, missingness
data_dictionary = pd.DataFrame({
    "column": prices.columns,

    "dtype": [str(prices[col].dtype) for col in prices.columns],

    "n_unique": [prices[col].nunique() for col in prices.columns],

    "n_missing": [prices[col].isna().sum() for col in prices.columns],

    "pct_missing": [(prices[col].isna().mean() * 100).round(2) for col in prices.columns],
})

data_dictionary

,column,dtype,n_unique,n_missing,pct_missing
0,date,datetime64[us],248,0,0.00
1,admin1,str,7,68,0.24
2,admin2,str,27,68,0.24
3,market,str,226,0,0.00
4,market_id,int64,226,0,0.00
5,latitude,float64,182,68,0.24
6,longitude,float64,184,68,0.24
7,category,str,8,0,0.00
8,commodity,str,51,0,0.00
9,commodity_id,int64,51,0,0.00


In [13]:
# scale and spread of the two numeric price fields

prices[["price", "usdprice"]].describe()

,price,usdprice
count,28062.000000,28062.000000
mean,1303.493627,11.921261
std,2567.718662,22.840086
min,5.000000,0.039000
25%,70.000000,0.610000
50%,130.000000,1.030000
75%,1007.430000,9.397500
max,19800.000000,184.820000


In [14]:
# check for duplicate rows at the expected grain

grain_columns = ["date", "market", "commodity", "pricetype"]

duplicate_count = prices.duplicated(subset=grain_columns).sum()

In [15]:
print(f"Duplicate rows at (date, market, commodity, pricetype) grain: {duplicate_count}")
print(f"Unique markets: {prices['market'].nunique()}")
print(f"Unique commodities: {prices['commodity'].nunique()}")
print(f"Unique admin1 regions: {prices['admin1'].nunique()}")

Duplicate rows at (date, market, commodity, pricetype) grain: 0
Unique markets: 226
Unique commodities: 51
Unique admin1 regions: 7


### 2.3 Exploratory Data Analysis

Exploratory analysis is used to understand the main patterns in the price and weather data.

We first examine the individual variables to understand their distributions, ranges, and data coverage. We then look at relationships between variables, with a focus on how weather conditions relate to commodity prices.
